# Module 6 - California workforce data cleaning

**Yixiang Lu | Vision A | Codex-assisted individual work**

Read `cleaning_plan.md` before executing. The plan was written after inspecting
original inputs and before applying transformations. This notebook runs the same
submitted Python implementation and an independent source validator; it does not
depend on hidden interactive variables or manual spreadsheet edits.

The unit of the main output is one California OEWS area x detailed occupation.
Published state-level Claude usage is contextual and the occupation-code mapping
remains provisional. It is not an exposure measure, local adoption rate, or investment score.

**Restart Kernel and Run All** after installing `requirements.txt` and arranging
the untouched originals as specified in `README.md`. Set `DSO576_SOURCE_ROOT`
to your original data folder, or put the originals under `raw/`.


In [1]:
import os
from pathlib import Path
import json
from clean_data import run
from validate_data import validate

SOURCE_ROOT = Path(os.environ.get("DSO576_SOURCE_ROOT", "raw")).resolve()
OUTPUT_DIR = Path(os.environ.get("DSO576_OUTPUT_DIR", "outputs")).resolve()
assert SOURCE_ROOT.is_dir(), "Set DSO576_SOURCE_ROOT or populate raw/ as described in README.md"
print("Source folder exists:", SOURCE_ROOT.is_dir())
print("Plan:", Path("cleaning_plan.md").read_text(encoding="utf-8"))

Source folder exists: True
Plan: # Cleaning plan - written before implementation

Author: Yixiang Lu, with Codex assistance. Project: AI Workforce Investment Planner,
Vision A (confirmed by the student). This is an individual submission; team
reconciliation is pending. Rules are proposed analytical choices for student review.

## Question and scope

Which California OEWS area-occupation combinations have substantial published
employment and merit further review alongside California occupation-level Claude
usage? Compare employment, median annual wages, and published AI-use shares
separately. Do not produce an investment score, ROI, or job-loss prediction.

Use both original June 26, 2026 Anthropic CSVs, the original May 2025 BLS metro
and nonmetro ZIP, and four original OpenAI Signals tables (all-message IWA,
work-message IWA, state rankings, state topic shares). Other Signals demographic
tables are outside this question. Inspect every row of each selected source.

## Initial observati

## Execute the declared rules on the full data

The script profiles all selected source rows; preserves source hashes, values,
suppression markers and record references; filters the declared scope; and checks
logical keys, conversions, ranges, and merge cardinality. Check results are written
to JSON/CSV, not inferred from a successful-looking sample.

In [2]:
checks = run(SOURCE_ROOT, OUTPUT_DIR)
checks["outputs"]["summary"]

claude_ai: inspected 1,636,573; retained 78,747


1p_api: inspected 491,705; retained 70,674


oesm25ma/BOS_M2025_dl.xlsx: inspected 48,689; retained 973


oesm25ma/MSA_M2025_dl.xlsx: inspected 150,023; retained 10,874


{
  "bls_rows": 11847,
  "areas": 28,
  "occupations": 783,
  "anthropic_rows": 149421,
  "employment_missing": 215,
  "wage_missing": 169,
  "employment_published_sum": 17277510.0,
  "employment_n": 11632,
  "wage_n": 11678,
  "unmatched_ai_rows": 5387
}


{'bls_rows': 11847,
 'areas': 28,
 'occupations': 783,
 'anthropic_rows': 149421,
 'employment_missing': 215,
 'wage_missing': 169,
 'employment_published_sum': 17277510.0,
 'employment_n': 11632,
 'wage_n': 11678,
 'unmatched_ai_rows': 5387}

## Independently verify the original-to-cleaned values

The validator reads the untouched workbooks with openpyxl and recomputes the
Claude shares with csv/Decimal. It does not call the cleaner's transformation
functions. Every selected BLS row and every joined AI value is checked, along
with the sample and five difficult record examples.

In [3]:
verification = validate(SOURCE_ROOT, OUTPUT_DIR)
assert verification["passed"]
verification["five_record_checks"]

{
  "passed": true,
  "independent_bls_rows_checked": 11847,
  "source_CA_rows_by_workbook": {
    "oesm25ma/BOS_M2025_dl.xlsx": 973,
    "oesm25ma/MSA_M2025_dl.xlsx": 10874
  },
  "numeric_columns_checked_per_row": 19,
  "employment_total_from_original": "17277510",
  "aei_leaf_rows_independently_aggregated": 460,
  "all_joined_rows_checked_against_original_AI": 11847,
  "sample_rows_exactly_match_full_output": 50
}


[{'AREA': '0600003',
  'OCC_CODE': '27-3041',
  'occupation': 'Editors',
  'source_file': 'oesm25ma/BOS_M2025_dl.xlsx',
  'source_record': 3353,
  'original': {'TOT_EMP': '**',
   'A_MEDIAN': '71300',
   'AREA': '0600003',
   'OCC_CODE': '27-3041'},
  'expected': {'TOT_EMP': None,
   'A_MEDIAN': '71300',
   'ca_claude_usage_pct': '2.49'},
  'actual': {'TOT_EMP': None,
   'A_MEDIAN': '71300.0',
   'ca_claude_usage_pct': '2.49'},
  'reason': 'Unpublished employment stays missing, not zero.',
  'matches': True},
 {'AREA': '0600006',
  'OCC_CODE': '29-1211',
  'occupation': 'Anesthesiologists',
  'source_file': 'oesm25ma/BOS_M2025_dl.xlsx',
  'source_record': 3699,
  'original': {'TOT_EMP': '**',
   'A_MEDIAN': '#',
   'AREA': '0600006',
   'OCC_CODE': '29-1211'},
  'expected': {'TOT_EMP': None, 'A_MEDIAN': None, 'ca_claude_usage_pct': None},
  'actual': {'TOT_EMP': None, 'A_MEDIAN': None, 'ca_claude_usage_pct': None},
  'reason': 'Unpublished employment stays missing, not zero. Top-coded 

## Inspect the sample and full-data evidence

The sample contains all cleaned columns and deliberately includes difficult cases.
It is not statistically representative. Numeric missing values are displayed as
missing, while the original suppression markers remain in `_raw` columns.

In [4]:
import pandas as pd
sample = pd.read_csv(OUTPUT_DIR / "cleaned_sample.csv", dtype={"AREA": str, "OCC_CODE": str, "NAICS": str})
display(sample[["AREA", "OCC_CODE", "TOT_EMP_raw", "TOT_EMP", "A_MEDIAN_raw", "A_MEDIAN", "ca_claude_usage_pct", "ai_match_status"]].head(10))
checks["merges"]

,AREA,OCC_CODE,TOT_EMP_raw,TOT_EMP,A_MEDIAN_raw,A_MEDIAN,ca_claude_usage_pct,ai_match_status
0,0600003,27-3041,**,NaN,71300,71300.0,2.49,both
1,0600006,29-1211,**,NaN,#,NaN,NaN,left_only
2,0600003,19-4042,90,90.0,*,NaN,NaN,left_only
3,0600003,11-9021,190,190.0,117310,117310.0,NaN,left_only
4,0600003,11-3031,380,380.0,130990,130990.0,0.41,both
5,0600003,11-1011,270,270.0,134650,134650.0,0.16,both
6,0600003,11-1021,1910,1910.0,97570,97570.0,0.17,both
7,0600003,11-2021,90,90.0,136730,136730.0,0.43,both
8,0600003,11-2022,490,490.0,87790,87790.0,0.14,both
9,0600003,11-3012,220,220.0,98210,98210.0,0.15,both


{'bls_to_ca_claude': {'key': 'OCC_CODE',
  'relationship': 'many BLS areas to one CA occupation prefix',
  'bls_before': 11847,
  'aei_leaf_before': 460,
  'aei_prefix_after': 401,
  'joined_after': 11847,
  'unmatched_bls_rows': 5387,
  'unmatched_bls_codes': 415,
  'unused_aei_prefixes': 33,
  'employment_before': 17277510.0,
  'employment_after': 17277510.0,
  'aei_share_before': 97.82,
  'aei_share_after': 97.82,
  'taxonomy_status': 'provisional; team must verify official crosswalk before decision use'}}

## Interpretation and unresolved decisions

- An absent published AI cell is not a zero-use occupation.
- The AI context is California-wide; employment is area-specific and from a different vintage.
- SOC-prefix aggregation needs an official taxonomy-crosswalk review.
- Published detailed employment sums omit suppressed cells. An unweighted mean
  of occupation medians is not an employee-level median wage.
- No ROI, causal effect or priority score follows from these cleaned inputs.
- Team reconciliation and approval of the final group report remain pending.

Reproduction evidence is generated by comparing this fresh-kernel run's cleaned
file hashes with the first process run. See `outputs/evidence/reproducibility.json`.
